# Heterogeneous Evolution Strategies for Mixed-Variable Optimization

## Experiment 1: Equal-Budget Benchmark Comparison

> Prototype 1 vs Baseline Optimizers (Isotropic ES and CMA-ES)

> This notebook compares Prototype 1 (our custom heterogeneous ES model) against the following baseline implementations:
1. **Isotropic ES (Simple Self-Adaptive)**: Uses a single global $\sigma$ ($n_\sigma = 1$, $n_\alpha = 0$).
2. **sep-CMA-ES (Separable CMA-ES)**: Diagonal CMA-ES from the `pycma` library (`CMA_diagonal=True`).
3. **Standard CMA-ES (Upper Bound)**: Full-covariance CMA-ES used strictly as a performance ceiling.

All comparisons use the **real-valued block only** for fairness, evaluated on
standard benchmark functions with equal evaluation budgets and 10 seeds.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
import cma

# Add project root to path so we can import project packages
sys.path.insert(0, os.path.abspath('..'))
from src.es_model import config as cfg
from src.es_model import kernels
from src.es_model.engine import Individual, create_random_individual, clone_individual, mutate_individual, select_survivors
from src.benchmarks.functions import BENCHMARKS
from src.runners import custom_es, isotropic_es, sep_cma_es, standard_cma_es

## 2. Benchmark Functions

Each function accepts a 1-D numpy array of shape `(n,)` and returns a scalar.
Global minimum is 0.0 at the origin (or known point) for all functions.

In [4]:
# BENCHMARKS is imported from src.benchmarks.functions
print(f"Loaded {len(BENCHMARKS)} benchmark definitions from src.benchmarks")

Loaded 4 benchmark definitions from src.benchmarks


In [5]:
# --- Verify global minima ---
DIM = 2
for name, info in BENCHMARKS.items():
    x_opt = info['optimum'](DIM)
    val = info['fn'](x_opt)
    assert abs(val) < 1e-10, f"{name} failed: f(optimum) = {val}"
    print(f"  {name:12s} f(optimum) = {val:.2e} ✓")
print("\nAll benchmark functions verified.")

  Sphere       f(optimum) = 0.00e+00 ✓
  Rastrigin    f(optimum) = 0.00e+00 ✓
  Rosenbrock   f(optimum) = 0.00e+00 ✓
  Ackley       f(optimum) = 4.44e-16 ✓

All benchmark functions verified.


## 3. Runner Wrappers

Each runner is imported from `src.runners` and implements the standard interface:
`run(fn, dim, bounds, seed, max_evals, **kwargs) -> np.ndarray`
returning a 1-D array of best-fitness values indexed by evaluation count.

In [6]:
# --- All runners registry ---
RUNNERS = {
    'Prototype 1':                          custom_es.run,
    'Isotropic ES (Simple Self-Adaptive)':  isotropic_es.run,
    'sep-CMA-ES (Separable CMA-ES)':        sep_cma_es.run,
    'Standard CMA-ES (Upper Bound)':        standard_cma_es.run,
}


## 4. Experiment Execution

Run all optimizers on all benchmarks with 10 seeds each.

In [7]:
# Experiment parameters
DIM = 10
MAX_EVALS = 10_000
N_SEEDS = 10
SEEDS = list(range(1, N_SEEDS + 1))


In [8]:
def run_all_experiments(benchmarks, runners, dim, max_evals, seeds):
    results = {}
    for bname, binfo in benchmarks.items():
        results[bname] = {}
        for rname, runner in runners.items():
            traces = []
            for seed in seeds:
                trace = runner(binfo["fn"], dim, binfo["bounds"], seed, max_evals)
                traces.append(trace)
            results[bname][rname] = np.array(traces)
            median_final = np.median(results[bname][rname][:, -1])
            print(f"  {bname:12s} | {rname:36s} | median final: {median_final:.4e}")
    return results

print("Running experiments...\n")
results = run_all_experiments(BENCHMARKS, RUNNERS, DIM, MAX_EVALS, SEEDS)
print("\nAll experiments complete.")


Running experiments...



c:\Users\bejmo\Documents\documents\GitHub\TFG\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\bejmo\Documents\documents\GitHub\TFG\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


IndexError: list index out of range

## 5. Visualization

Convergence plots with median ± IQR for all optimizers on each benchmark.

In [ ]:
COLORS = {
    'Prototype 1':                          '#E63946',
    'Isotropic ES (Simple Self-Adaptive)':  '#E9C46A',
    'sep-CMA-ES (Separable CMA-ES)':        '#2A9D8F',
    'Standard CMA-ES (Upper Bound)':        '#457B9D',
}


def plot_convergence(benchmark_name: str, data: dict, max_evals: int) -> None:
    """
    Plot median convergence with IQR envelope for all optimizers.

    Parameters
    ----------
    benchmark_name : str
    data : dict[runner_name] = ndarray of shape (n_seeds, max_evals)
    max_evals : int
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    evals = np.arange(1, max_evals + 1)

    for rname, traces in data.items():
        median = np.median(traces, axis=0)
        q25 = np.percentile(traces, 25, axis=0)
        q75 = np.percentile(traces, 75, axis=0)

        color = COLORS.get(rname, '#888888')
        # Use dashed line for Upper Bound to differentiate it visually
        ls = '--' if 'Upper Bound' in rname else '-'
        ax.plot(evals, median, label=rname, color=color, linewidth=1.5, linestyle=ls)
        ax.fill_between(evals, q25, q75, alpha=0.15, color=color)

    ax.set_yscale('log')
    ax.set_xlabel('Evaluations')
    ax.set_ylabel('Best Fitness (log)')
    ax.set_title(f'Convergence Comparison — {benchmark_name}')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
for bname, data in results.items():
    plot_convergence(bname, data, MAX_EVALS)